<a href="https://colab.research.google.com/github/mizinco/cv-practice/blob/main/SAM2_observation_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Sun Jun  7 08:46:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd /content/sam2
!pip install -e . -q
!pip install opencv-python matplotlib -q

/content
Cloning into 'sam2'...
remote: Enumerating objects: 1107, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 1107 (delta 11), reused 2 (delta 2), pack-reused 1089 (from 2)
Receiving objects: 100% (1107/1107), 134.92 MiB | 21.03 MiB/s, done.
Resolving deltas: 100% (384/384), done.
/content/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 9.3 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done


In [3]:
%cd /content/sam2/checkpoints
!./download_ckpts.sh
!ls -lh /content/sam2/checkpoints/

/content/sam2/checkpoints
--2026-06-07 08:50:08--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.163.158.86, 3.163.158.84, 3.163.158.24, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.163.158.86|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 156008466 (149M) [application/vnd.snesdev-page-table]
Saving to: ‘sam2.1_hiera_tiny.pt’

sam2.1_hiera_tiny.p 100%[===================>] 148.78M  33.4MB/s    in 4.8s    

2026-06-07 08:50:13 (31.1 MB/s) - ‘sam2.1_hiera_tiny.pt’ saved [156008466/156008466]

--2026-06-07 08:50:13--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.163.158.86, 3.163.158.84, 3.163.158.24, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.163.158.86|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length:

In [4]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import cv2

sys.path.insert(0, '/content/sam2')
from sam2.build_sam import build_sam2_video_predictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.autocast(device_type=device.type, dtype=torch.bfloat16).__enter__()
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print('using device:', device)

using device: cuda


In [5]:
sam2_checkpoint = '/content/sam2/checkpoints/sam2.1_hiera_large.pt'
model_cfg = 'configs/sam2.1/sam2.1_hiera_l.yaml'
predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)
print('predictor ready')

predictor ready


In [6]:
import os
import json
import subprocess
import tempfile

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
import cv2
from PIL import Image
import gradio as gr


# ============================================================================
# 指標関数（src/metrics.py の内容を inline 化、Colab self-contain）
# ============================================================================

def mask_area_change_rate(masks):
    areas = masks.sum(axis=(1, 2)).astype(np.float64)
    change_rate = np.zeros(len(areas), dtype=np.float32)
    for t in range(1, len(areas)):
        prev, curr = areas[t - 1], areas[t]
        if prev == 0:
            change_rate[t] = 0.0 if curr == 0 else float('inf')
        else:
            change_rate[t] = (curr - prev) / prev
    return change_rate


def connected_components_count(masks, min_size=50):
    counts = np.zeros(len(masks), dtype=np.int32)
    for t in range(len(masks)):
        if masks[t].sum() == 0:
            counts[t] = 0
            continue
        labeled, n = ndimage.label(masks[t])
        if n == 0:
            counts[t] = 0
        elif min_size > 0:
            sizes = ndimage.sum(masks[t], labeled, range(1, n + 1))
            counts[t] = int((sizes >= min_size).sum())
        else:
            counts[t] = n
    return counts


def largest_component_ratio(masks):
    ratios = np.ones(len(masks), dtype=np.float32)
    for t in range(len(masks)):
        total = masks[t].sum()
        if total == 0:
            ratios[t] = 1.0
            continue
        labeled, n = ndimage.label(masks[t])
        if n == 0:
            ratios[t] = 1.0
            continue
        sizes = ndimage.sum(masks[t], labeled, range(1, n + 1))
        ratios[t] = float(sizes.max() / total)
    return ratios


def shape_iou_temporal(masks):
    T = len(masks)
    iou = np.ones(T, dtype=np.float32)
    if T == 0:
        return iou
    H, W = masks[0].shape

    def _centroid(m):
        ys, xs = np.where(m)
        if len(ys) == 0:
            return None
        return float(ys.mean()), float(xs.mean())

    for t in range(1, T):
        m_prev, m_curr = masks[t - 1], masks[t]
        if m_prev.sum() == 0 or m_curr.sum() == 0:
            iou[t] = 1.0
            continue
        c_prev = _centroid(m_prev)
        c_curr = _centroid(m_curr)
        dy = int(round(c_prev[0] - c_curr[0]))
        dx = int(round(c_prev[1] - c_curr[1]))
        shifted = np.zeros_like(m_curr)
        ys, ye = max(0, dy), min(H, H + dy)
        xs, xe = max(0, dx), min(W, W + dx)
        if ys < ye and xs < xe:
            shifted[ys:ye, xs:xe] = m_curr[ys - dy:ye - dy, xs - dx:xe - dx]
        inter = np.logical_and(m_prev, shifted).sum()
        union = np.logical_or(m_prev, shifted).sum()
        iou[t] = float(inter / union) if union > 0 else 1.0
    return iou


def mask_disappearance(masks, threshold_abs=50, threshold_rel=0.01, ref_frames=5):
    areas = masks.sum(axis=(1, 2)).astype(np.float64)
    ref = areas[:max(ref_frames, 1)]
    ref_area = float(ref.mean()) if len(ref) > 0 else 1.0
    threshold = max(float(threshold_abs), ref_area * threshold_rel)
    is_dis = areas < threshold
    cons = np.zeros(len(areas), dtype=np.int32)
    for t in range(len(areas)):
        if is_dis[t]:
            cons[t] = (cons[t - 1] if t > 0 else 0) + 1
        else:
            cons[t] = 0
    return {
        'areas': areas,
        'is_disappeared': is_dis,
        'consecutive': cons,
        'threshold': threshold,
        'reference_area': ref_area,
    }


def compute_all_metrics(masks, threshold_abs=50, min_size=50):
    area_change = mask_area_change_rate(masks)
    components = connected_components_count(masks, min_size=min_size)
    largest = largest_component_ratio(masks)
    disappear = mask_disappearance(masks, threshold_abs=threshold_abs)
    shape_iou = shape_iou_temporal(masks)
    return {
        'frame_count': int(len(masks)),
        'areas': disappear['areas'].astype(float).tolist(),
        'area_change_rate': area_change.tolist(),
        'connected_components': components.tolist(),
        'largest_component_ratio': largest.tolist(),
        'shape_iou_temporal': shape_iou.tolist(),
        'is_disappeared': disappear['is_disappeared'].tolist(),
        'consecutive_disappearance': disappear['consecutive'].tolist(),
        'reference_area': float(disappear['reference_area']),
        'disappearance_threshold': float(disappear['threshold']),
        'min_component_size': int(min_size),
    }


# ============================================================================
# 5パネル可視化（修正済み順序: 面積 → 変化率 → 成分 → 消失 → 形状 IoU）
# ============================================================================

def make_5panel_plot(metrics, case_name='session'):
    plt.rcParams['font.family'] = 'DejaVu Sans'
    fig, axes = plt.subplots(5, 1, figsize=(11, 12), sharex=True)
    frames = np.arange(metrics['frame_count'])

    # 0. Mask area + threshold
    axes[0].plot(frames, metrics['areas'], color='steelblue', linewidth=1.5)
    axes[0].axhline(
        metrics['disappearance_threshold'], color='red', linestyle='--',
        label=f"Disappearance threshold ({metrics['disappearance_threshold']:.0f}px)",
    )
    axes[0].set_ylabel('Mask area (px)')
    axes[0].set_title(f'{case_name} - mask area over time')
    axes[0].legend(loc='upper right')
    axes[0].grid(alpha=0.3)

    # 1. Area change rate (Indicator 1)
    rates = np.array(metrics['area_change_rate'], dtype=float)
    rates_disp = np.clip(rates, -2.0, 2.0)
    axes[1].plot(frames, rates_disp, color='darkorange', linewidth=1.5)
    axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='+-50%')
    axes[1].axhline(-0.5, color='red', linestyle='--', alpha=0.5)
    axes[1].set_ylabel('Area change rate')
    axes[1].set_title('Indicator 1: Mask area change rate (clipped to +-2.0)')
    axes[1].legend(loc='upper right')
    axes[1].grid(alpha=0.3)

    # 2. Components (Indicator 2)
    axes[2].plot(
        frames, metrics['connected_components'],
        color='seagreen', linewidth=1.5, marker='o', markersize=3,
    )
    axes[2].set_ylabel('Component count')
    axes[2].set_title(
        f'Indicator 2: Connected components (>={metrics["min_component_size"]}px only)'
    )
    axes[2].grid(alpha=0.3)

    # 3. Disappearance (Indicator 3)
    is_dis = np.array(metrics['is_disappeared'])
    cons = np.array(metrics['consecutive_disappearance'])
    axes[3].fill_between(
        frames, 0, is_dis.astype(int),
        alpha=0.4, color='crimson', step='post', label='Disappeared',
    )
    if cons.max() > 0:
        axes[3].plot(
            frames, cons / max(cons.max(), 1),
            color='crimson', linewidth=1.5, label='Consecutive (normalized)',
        )
    axes[3].set_ylim(-0.05, 1.1)
    axes[3].set_ylabel('Disappearance')
    axes[3].set_title(
        f'Indicator 3: Mask disappearance (max consecutive: {int(cons.max())} frames)'
    )
    axes[3].legend(loc='upper left')
    axes[3].grid(alpha=0.3)

    # 4. Shape IoU (Indicator 4)
    siou = np.array(metrics['shape_iou_temporal'], dtype=float)
    axes[4].plot(frames, siou, color='purple', linewidth=1.5)
    axes[4].axhline(0.8, color='red', linestyle='--', alpha=0.5, label='IoU 0.8')
    axes[4].set_ylim(0, 1.05)
    min_idx = int(np.argmin(siou))
    axes[4].set_title(
        f'Indicator 4: Shape stability (centroid-aligned IoU, min '
        f'{float(siou.min()):.3f} at frame {min_idx})'
    )
    axes[4].set_ylabel('Shape IoU')
    axes[4].set_xlabel('Frame')
    axes[4].legend(loc='lower right')
    axes[4].grid(alpha=0.3)

    plt.tight_layout()
    return fig


# ============================================================================
# ★ フォーカス動画生成
# ============================================================================
# 各フレームのマスク重心を中心とした固定サイズの窓で切り出し、自動追従カメラワーク
# を生成する。形状 IoU < 閾値 または消失中のフレームには赤枠と LOW QUALITY ラベル
# を焼き込む（品質警告）。
#
# 使い方: tracking 実行後に「Generate focus video」ボタンを押す。
# 出力: focus_video.mp4 を out_dir に保存、Gradio から再生 + ダウンロード可能。
# ============================================================================

def make_focus_video(
    state,
    output_size: int = 400,
    constant_ratio: bool = True,
    target_mask_ratio: float = 0.3,
    iou_warning_threshold: float = 0.5,
    show_mask_overlay: bool = True,
    progress=gr.Progress(),
):
    """
    フォーカス動画を生成する。

    Args:
        output_size: 出力フレームサイズ（正方形、ピクセル）
        constant_ratio: True なら「画面内マスク比一定」モード。
                        False なら固定ウィンドウサイズ（output_size と同じ）。
        target_mask_ratio: マスクが出力フレームに占める目標比率（constant_ratio=True 時のみ有効）。
                           マスク面積 / ウィンドウ面積 = target_mask_ratio となるようウィンドウサイズを毎フレーム計算。
        iou_warning_threshold: 形状 IoU がこれを下回るフレームに赤枠 + LOW QUALITY ラベルを焼き込む。
        show_mask_overlay: True なら緑半透明のマスクオーバーレイをフォーカス動画に重ねる。
                           破綻フレームで断片化や形状崩壊が視認可能。
                           False ならオーバーレイなしのきれいなクロップ動画（PPT デモ用）。

    アルゴリズム:
        1. 各フレームのマスク面積から、必要なウィンドウサイズを計算
           window_size_t = sqrt(mask_area_t / target_mask_ratio)
        2. EMA（α=0.15、時定数 ≈ 7 フレーム）で平滑化してジッターを抑制
        3. クランプ: 100px 以上、min(H, W) 以下
        4. マスクが空のフレームは前フレームのウィンドウサイズと重心を保持
        5. クロップ → output_size × output_size にリサイズ
    """
    if state is None or state.get('masks') is None:
        return None, None, 'Run tracking first to generate focus video'

    SMOOTHING_ALPHA = 0.15      # EMA 係数（小さいほど滑らか、大きいほど追従性高）
    MIN_WINDOW_SIZE = 100       # 過度なズームインを防ぐ下限
    output_size = int(output_size)
    target_mask_ratio = float(target_mask_ratio)
    iou_warning_threshold = float(iou_warning_threshold)

    masks = state['masks']
    frame_dir = state['frame_dir']
    out_dir = state['out_dir']

    # 最新の metrics.json から指標を読む（スライダーで再計算後の値）
    metrics_json_path = state.get('metrics_json')
    if metrics_json_path and os.path.exists(metrics_json_path):
        with open(metrics_json_path) as f:
            metrics = json.load(f)
        siou = np.array(metrics.get('shape_iou_temporal', []), dtype=float)
        is_dis = np.array(metrics.get('is_disappeared', []), dtype=bool)
    else:
        siou = np.ones(len(masks), dtype=float)
        is_dis = np.zeros(len(masks), dtype=bool)

    frames_list = sorted(os.listdir(frame_dir))
    H_full, W_full = state['first_frame'].shape[:2]
    max_window = min(H_full, W_full)

    focus_dir = os.path.join(out_dir, 'focus')
    os.makedirs(focus_dir, exist_ok=True)

    # --- パス 1: ウィンドウサイズの時系列を計算 + EMA 平滑化 ----------------
    raw_window_sizes = []
    last_ws = output_size
    for i in range(len(masks)):
        m = masks[i]
        if m.any():
            mask_area = float(m.sum())
            if constant_ratio:
                # mask_area / ws^2 == target_ratio → ws = sqrt(mask_area / target_ratio)
                ws = int(round(np.sqrt(mask_area / max(target_mask_ratio, 0.01))))
            else:
                ws = output_size
            ws = max(MIN_WINDOW_SIZE, min(max_window, ws))
            last_ws = ws
        else:
            ws = last_ws  # 消失時は前を保持
        raw_window_sizes.append(ws)

    # EMA 平滑化
    smoothed_sizes = []
    ema = float(raw_window_sizes[0]) if raw_window_sizes else float(output_size)
    for ws in raw_window_sizes:
        ema = SMOOTHING_ALPHA * float(ws) + (1.0 - SMOOTHING_ALPHA) * ema
        smoothed_sizes.append(int(round(ema)))

    # --- パス 2: 各フレームをクロップ + リサイズ ----------------------------
    last_cy, last_cx = H_full // 2, W_full // 2  # 消失時のフォールバック

    progress(0, desc='Generating focus video')
    for i, fname in enumerate(frames_list):
        img = cv2.imread(os.path.join(frame_dir, fname))
        if img is None:
            continue

        # 重心計算（マスクがあればマスク重心、無ければ前回の重心を保持）
        m = masks[i] if i < len(masks) else None
        if m is not None and m.any():
            ys, xs = np.where(m)
            cy, cx = int(ys.mean()), int(xs.mean())
            last_cy, last_cx = cy, cx
        else:
            cy, cx = last_cy, last_cx

        ws = smoothed_sizes[i] if i < len(smoothed_sizes) else output_size
        half_h = ws // 2
        half_w = ws // 2

        # 窓のクロップ範囲（画像境界でクランプ）
        y0 = max(0, cy - half_h)
        y1 = min(H_full, cy + half_h)
        x0 = max(0, cx - half_w)
        x1 = min(W_full, cx + half_w)

        crop = img[y0:y1, x0:x1].copy()

        # ★ マスクオーバーレイ（破綻フレームで断片化が視認できる）
        if show_mask_overlay and m is not None:
            mask_crop = m[y0:y1, x0:x1]
            if mask_crop.any():
                ov = crop.copy()
                ov[mask_crop] = (0, 255, 0)
                crop = cv2.addWeighted(crop, 0.6, ov, 0.4, 0)

        # 境界クランプで縮んだ場合は黒 pad で ws×ws に揃える
        actual_h, actual_w = crop.shape[:2]
        if actual_h < ws or actual_w < ws:
            padded = np.zeros((ws, ws, 3), dtype=np.uint8)
            pad_y = (ws - actual_h) // 2
            pad_x = (ws - actual_w) // 2
            padded[pad_y:pad_y + actual_h, pad_x:pad_x + actual_w] = crop
            crop = padded

        # ★ 出力解像度にリサイズ（これでマスク比が一定になる）
        if crop.shape[0] != output_size or crop.shape[1] != output_size:
            crop = cv2.resize(crop, (output_size, output_size),
                              interpolation=cv2.INTER_LINEAR)

        # 品質警告: IoU < 閾値 または 消失中
        warn = False
        if i < len(siou) and float(siou[i]) < iou_warning_threshold:
            warn = True
        if i < len(is_dis) and bool(is_dis[i]):
            warn = True
        if warn:
            cv2.rectangle(crop, (0, 0), (output_size - 1, output_size - 1),
                          (0, 0, 255), 8)
            cv2.putText(crop, 'LOW QUALITY', (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # フレーム番号 + 現在のウィンドウサイズ（デバッグ用、左下）
        cv2.putText(crop, f'f={i} ws={ws}', (10, output_size - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        cv2.imwrite(os.path.join(focus_dir, f'{i:05d}.jpg'), crop)
        progress((i + 1) / len(frames_list),
                 desc=f'Focus frame {i + 1}/{len(frames_list)}')

    focus_mp4 = os.path.join(out_dir, 'focus_video.mp4')
    fps = state.get('target_fps', 10)
    subprocess.run([
        'ffmpeg', '-y', '-framerate', str(fps),
        '-i', os.path.join(focus_dir, '%05d.jpg'),
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', focus_mp4,
    ], check=True, capture_output=True)

    state['focus_mp4'] = focus_mp4

    # 警告フレーム数とウィンドウサイズ範囲の集計
    warn_count = 0
    for i in range(len(frames_list)):
        if (i < len(siou) and float(siou[i]) < iou_warning_threshold) or \
           (i < len(is_dis) and bool(is_dis[i])):
            warn_count += 1

    if smoothed_sizes:
        ws_min = min(smoothed_sizes)
        ws_max = max(smoothed_sizes)
        ws_mean = sum(smoothed_sizes) / len(smoothed_sizes)
        ws_info = f'window size: min={ws_min} mean={ws_mean:.0f} max={ws_max}'
    else:
        ws_info = ''

    mode_info = (
        f'constant ratio mode (target {target_mask_ratio:.2f})'
        if constant_ratio else
        f'fixed window mode ({output_size}px)'
    )

    summary = (
        f'Focus video generated: {len(frames_list)} frames, '
        f'output {output_size}x{output_size}px, {mode_info}. '
        f'{ws_info}. '
        f'Quality warnings: {warn_count} frames '
        f'(IoU<{iou_warning_threshold} or disappeared).'
    )
    return focus_mp4, focus_mp4, summary


# ============================================================================
# Gradio コールバック
# ============================================================================

def upload_video(video_path, target_fps, state):
    """
    動画をアップロード→ffmpeg でフレーム抽出。
    target_fps でダウンサンプルし、高 fps 動画（60fps スマホ動画等）でも
    SAM2 推論時間が爆発しないようにする。
    """
    if video_path is None:
        return None, state, 'Upload a video first'
    out_dir = tempfile.mkdtemp(prefix='gradio_frames_')
    target_fps = int(target_fps)
    # vf チェーン: fps ダウンサンプル → padding（H/W を偶数化）
    vf = f'fps={target_fps},pad=ceil(iw/2)*2:ceil(ih/2)*2'
    cmd = [
        'ffmpeg', '-y', '-i', video_path,
        '-vf', vf,
        '-q:v', '2', os.path.join(out_dir, '%05d.jpg'),
    ]
    subprocess.run(cmd, check=True, capture_output=True)
    frames = sorted(os.listdir(out_dir))
    if not frames:
        return None, state, 'No frames extracted'
    first = np.array(Image.open(os.path.join(out_dir, frames[0])))

    inference_state = predictor.init_state(video_path=out_dir)

    state = {
        'frame_dir': out_dir,
        'video_path': video_path,
        'n_frames': len(frames),
        'target_fps': target_fps,
        'first_frame': first,
        'click_xy': None,
        'masks': None,
        'tracked_mp4': None,
        'focus_mp4': None,
        'masks_npz': None,
        'metrics_json': None,
        'metrics_png': None,
        'out_dir': None,
        'inference_state': inference_state,
    }
    return (
        first, state,
        f'Loaded {len(frames)} frames @ {target_fps} fps '
        f'(downsampled from input). Click on the target in the image.'
    )


def on_click(state, evt: gr.SelectData):
    if state is None or 'first_frame' not in state:
        return None, state, 'Upload a video first'
    x, y = int(evt.index[0]), int(evt.index[1])
    state['click_xy'] = (x, y)
    img = state['first_frame'].copy()
    cv2.drawMarker(img, (x, y), (255, 0, 0),
                   markerType=cv2.MARKER_CROSS, markerSize=30, thickness=3)
    return img, state, f'Clicked at ({x}, {y}). Press "Run tracking".'


def run_tracking(state, thr_abs, min_comp, progress=gr.Progress()):
    if state is None or state.get('click_xy') is None:
        return None, None, None, None, None, state, 'Click a point first'

    inference_state = state['inference_state']
    predictor.reset_state(inference_state)
    x, y = state['click_xy']

    predictor.add_new_points_or_box(
        inference_state=inference_state, frame_idx=0, obj_id=1,
        points=np.array([[x, y]], dtype=np.float32),
        labels=np.array([1], dtype=np.int32),
    )

    frame_dir = state['frame_dir']
    frames_list = sorted(os.listdir(frame_dir))
    H, W = state['first_frame'].shape[:2]
    masks_all = np.zeros((len(frames_list), H, W), dtype=bool)

    progress(0, desc='Propagating')
    for fidx, _obj_ids, mask_logits in predictor.propagate_in_video(inference_state):
        if fidx < len(frames_list):
            m = (mask_logits[0] > 0.0).cpu().numpy()[0]
            masks_all[fidx] = m
        progress((fidx + 1) / len(frames_list),
                 desc=f'Frame {fidx + 1}/{len(frames_list)}')

    # オーバーレイ MP4
    out_dir = tempfile.mkdtemp(prefix='gradio_out_')
    overlay_dir = os.path.join(out_dir, 'overlay')
    os.makedirs(overlay_dir, exist_ok=True)
    for i, fname in enumerate(frames_list):
        img = cv2.imread(os.path.join(frame_dir, fname))
        if masks_all[i].any():
            overlay = img.copy()
            overlay[masks_all[i]] = (0, 255, 0)
            img = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)
        cv2.imwrite(os.path.join(overlay_dir, f'{i:05d}.jpg'), img)

    tracked_mp4 = os.path.join(out_dir, 'tracked.mp4')
    fps = state.get('target_fps', 10)
    subprocess.run([
        'ffmpeg', '-y', '-framerate', str(fps),
        '-i', os.path.join(overlay_dir, '%05d.jpg'),
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', tracked_mp4,
    ], check=True, capture_output=True)

    # masks.npz
    masks_npz = os.path.join(out_dir, 'masks.npz')
    np.savez_compressed(
        masks_npz, masks=masks_all,
        click_xy=np.array([x, y]), case_name='session',
    )

    # metrics
    metrics = compute_all_metrics(
        masks_all, threshold_abs=int(thr_abs), min_size=int(min_comp),
    )
    metrics_json = os.path.join(out_dir, 'metrics.json')
    with open(metrics_json, 'w') as f:
        json.dump(metrics, f, indent=2)

    fig = make_5panel_plot(metrics, case_name='session')
    metrics_png = os.path.join(out_dir, 'metrics.png')
    fig.savefig(metrics_png, dpi=120, bbox_inches='tight')

    state['masks'] = masks_all
    state['tracked_mp4'] = tracked_mp4
    state['masks_npz'] = masks_npz
    state['metrics_json'] = metrics_json
    state['metrics_png'] = metrics_png
    state['out_dir'] = out_dir
    state['focus_mp4'] = None  # 新規追跡なのでフォーカス動画はリセット

    n_with_mask = int((masks_all.sum(axis=(1, 2)) > 0).sum())
    siou_arr = np.array(metrics['shape_iou_temporal'], dtype=float)
    summary = (
        f'Tracking done: {n_with_mask}/{len(frames_list)} frames with mask. '
        f'Click: ({x}, {y}). '
        f'Min shape IoU: {float(siou_arr.min()):.3f} at frame {int(np.argmin(siou_arr))}. '
        f'Disappeared frames: {sum(metrics["is_disappeared"])}, '
        f'max consecutive: {max(metrics["consecutive_disappearance"])}. '
        f'Output dir: {out_dir}'
    )
    return tracked_mp4, fig, masks_npz, metrics_json, metrics_png, state, summary


def recompute_metrics(state, thr_abs, min_comp):
    if state is None or state.get('masks') is None:
        return None, None, None, 'Run tracking first'
    masks_all = state['masks']
    metrics = compute_all_metrics(
        masks_all, threshold_abs=int(thr_abs), min_size=int(min_comp),
    )
    out_dir = state['out_dir']
    metrics_json = os.path.join(out_dir, 'metrics.json')
    with open(metrics_json, 'w') as f:
        json.dump(metrics, f, indent=2)
    fig = make_5panel_plot(metrics, case_name='session')
    metrics_png = os.path.join(out_dir, 'metrics.png')
    fig.savefig(metrics_png, dpi=120, bbox_inches='tight')
    state['metrics_json'] = metrics_json
    state['metrics_png'] = metrics_png
    summary = (
        f'Recomputed. Threshold={int(thr_abs)}px, min_comp={int(min_comp)}px. '
        f'Disappeared frames: {sum(metrics["is_disappeared"])}, '
        f'max consecutive: {max(metrics["consecutive_disappearance"])}.'
    )
    return fig, metrics_json, metrics_png, summary


def generate_focus_video_callback(
    state, output_size, constant_ratio, target_mask_ratio,
    iou_warning_threshold, show_mask_overlay,
):
    return make_focus_video(
        state,
        output_size=int(output_size),
        constant_ratio=bool(constant_ratio),
        target_mask_ratio=float(target_mask_ratio),
        iou_warning_threshold=float(iou_warning_threshold),
        show_mask_overlay=bool(show_mask_overlay),
    )


# ============================================================================
# UI
# ============================================================================

with gr.Blocks(title='SAM2 Recognition Limit Observer') as demo:
    gr.Markdown(
        '# SAM2 Recognition Limit Observer\n'
        'アップロードした映像に SAM2 動画追跡を適用し、マスクの破綻を 4 指標で可視化、'
        '対象追従カメラワーク（フォーカス動画）を自動生成します。'
    )
    state = gr.State(value=None)

    # ── 1〜3: 追跡フロー ────────────────────────────────────────
    gr.Markdown('## 1. 追跡フロー')
    with gr.Row():
        with gr.Column(scale=1):
            video_in = gr.Video(label='Input video', sources=['upload'])
            target_fps_slider = gr.Slider(
                5, 30, value=10, step=1,
                label='Extract FPS (downsample input)',
                info='高 fps 動画（60fps スマホ等）でも 10fps で抽出すれば速くなる',
            )
            upload_btn = gr.Button('1. Load video', variant='secondary')
            first_frame = gr.Image(
                label='2. Click on the target',
                type='numpy', interactive=False,
            )
        with gr.Column(scale=1):
            thr_abs = gr.Slider(
                0, 1000, value=50, step=10,
                label='Disappearance threshold (abs px)',
            )
            min_comp = gr.Slider(
                0, 500, value=50, step=10,
                label='Min component size (px)',
            )
            track_btn = gr.Button('3. Run tracking', variant='primary')
            recompute_btn = gr.Button(
                '4. Recompute metrics (no re-track)', variant='secondary',
            )
            tracked_video = gr.Video(label='Tracked output (overlay)')

    status = gr.Textbox(label='Status', value='Ready', lines=2)
    metrics_plot = gr.Plot(label='Metrics (5 panels)')

    # ── 4: フォーカス動画 ───────────────────────────────────────
    gr.Markdown(
        '## 2. フォーカス動画\n'
        '追跡結果のマスク重心を追従する自動クロップ動画を生成します。'
        '「定比モード」では画面内のマスク比率が一定になるよう、'
        'フレームごとにウィンドウサイズを動的調整します。'
        '形状 IoU が閾値を下回るフレームや消失フレームには赤枠と '
        '"LOW QUALITY" ラベルをつけて、品質を可視化します。'
    )
    with gr.Row():
        with gr.Column(scale=1):
            output_size_slider = gr.Slider(
                200, 800, value=400, step=50,
                label='Output frame size (px, square)',
            )
            constant_ratio_checkbox = gr.Checkbox(
                value=True,
                label='Constant mask ratio mode (動的ズーム)',
                info='オン: マスク比率が一定になるようウィンドウサイズを毎フレーム調整。オフ: 固定ウィンドウサイズ。',
            )
            target_ratio_slider = gr.Slider(
                0.05, 0.6, value=0.3, step=0.05,
                label='Target mask ratio (constant ratio mode のみ)',
                info='マスクが画面内に占める目標比率。0.3 なら 30%。',
            )
            iou_warn_slider = gr.Slider(
                0.0, 1.0, value=0.5, step=0.05,
                label='Quality warning IoU threshold',
            )
            show_mask_checkbox = gr.Checkbox(
                value=True,
                label='Show mask overlay on focus video',
                info='オン: 緑半透明マスクを焼き込み、破綻フレームで断片化を視認可能。オフ: クリーンなクロップ動画。',
            )
            focus_btn = gr.Button('5. Generate focus video', variant='primary')
        with gr.Column(scale=1):
            focus_video_display = gr.Video(label='Focus video output')

    # ── 5: ダウンロード ────────────────────────────────────────
    gr.Markdown('## 3. ダウンロード')
    with gr.Row():
        dl_tracked = gr.File(label='tracked.mp4')
        dl_focus = gr.File(label='focus_video.mp4')
        dl_npz = gr.File(label='masks.npz')
        dl_json = gr.File(label='metrics.json')
        dl_png = gr.File(label='metrics.png')

    # ── イベント結線 ────────────────────────────────────────────
    upload_btn.click(
        upload_video, [video_in, target_fps_slider, state],
        [first_frame, state, status],
    )
    first_frame.select(
        on_click, [state],
        [first_frame, state, status],
    )
    track_btn.click(
        run_tracking, [state, thr_abs, min_comp],
        [tracked_video, metrics_plot, dl_npz, dl_json, dl_png, state, status],
    ).then(
        lambda s: s.get('tracked_mp4') if s else None,
        [state], [dl_tracked],
    )
    recompute_btn.click(
        recompute_metrics, [state, thr_abs, min_comp],
        [metrics_plot, dl_json, dl_png, status],
    )
    focus_btn.click(
        generate_focus_video_callback,
        [state, output_size_slider, constant_ratio_checkbox,
         target_ratio_slider, iou_warn_slider, show_mask_checkbox],
        [focus_video_display, dl_focus, status],
    )

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d372954aaa7f04414f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
